# WESAD HSSL+DPBL Stress Detection Pipeline

**Entry point for Kaggle.**

This notebook runs the full pipeline in stages. Each stage checks for existing checkpoints
so if Kaggle times out (12h limit), you can re-run and it will resume from where it left off.

---
## Setup
1. **Runtime** → Change runtime type → **GPU** (T4/P100 recommended)
2. **Add Data** → Search `wesad-wearable-stress-affect-detection-dataset` → Add
3. **Run All** (or run cells sequentially)

---

## Stage 0 — Clone Repo & Install Dependencies

In [ ]:
import os, sys, subprocess

WORKING = "/kaggle/working"
os.chdir(WORKING)

# Clone repo if not already cloned
REPO_URL = "https://github.com/Notdet3cted/hssl-plus-dpbl.git"
REPO_DIR = os.path.join(WORKING, "hssl-plus-dpbl")

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print("Repository already cloned.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies (Kaggle already has most)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)
print("Dependencies installed.")

In [ ]:
# Run Kaggle setup: update config.yaml paths to /kaggle/working/
subprocess.run([sys.executable, "kaggle_setup.py", "--skip-install"], check=True)
print("Kaggle path setup complete.")

---
## Stage 1 — Data Preparation

*Acquisition → Preprocessing → LOSO folds → Windowing*

Skips if outputs already exist.

In [ ]:
!python run_kaggle.py --mode data_prep --skip-setup

---
## Stage 2 — SSL Pre-training & Embeddings

Pre-train SimSiam SSL encoder, then generate SSL embeddings.

*Note: This can take ~1-2h on Kaggle GPU.*

In [ ]:
!python run_kaggle.py --mode server_ssl --ssl_epochs 30 --skip-setup

---
## Stage 3 — Baseline Classifiers (RF & CNN) + SSL Classifier

In [ ]:
!python run_kaggle.py --mode server_a --epochs 20 --skip-setup

---
## Stage 4 — HSSL Pre-training, Embeddings & Classifier

In [ ]:
!python run_kaggle.py --mode server_b --epochs 20 --skip-setup

---
## Stage 5 — DPBL Training, Embeddings & HSSL+DPBL / SSL+DPBL Classifiers

Requires HSSL checkpoints from Stage 4.

In [ ]:
!python run_kaggle.py --mode server_c --epochs 20 --skip-setup

---
## Stage 6 — Robustness Testing

Tests the best model 5 times (reduced for Kaggle) to verify stability.

In [ ]:
!python run_kaggle.py --mode server_d --robust_iter 5 --epochs 20 --skip-setup

---
## Stage 7 — Evaluation, Comparison, Stats & Dashboard

Final aggregation. Produces `interactive_dashboard.html`.

In [ ]:
!python run_kaggle.py --mode eval --skip-setup

---
## Results

Outputs are in `/kaggle/working/hssl-plus-dpbl/results/`

**Dashboard:** `results/interactive_dashboard.html` → download & open in browser.

---

### Troubleshooting

| Problem | Solution |
|---------|----------|
| Kaggle timeout after 12h | Re-run **Run All**. Completed stages will be skipped automatically. |
| Out of GPU memory | Restart runtime with **Accelerator=None** (CPU-only), or reduce `batch_size` in `config/config.yaml` |
| WESAD dataset not found | Check **Add Data** → search `wesad-wearable-stress-affect-detection-dataset` |
| Missing checkpoint error | Run stages sequentially; some stages depend on previous ones. |